# V1
来自baseline, 针对测试得到的问题进行处理分析

**问题**：md每个都是菜谱，南派红烧肉，徽派红烧肉，家常红烧肉，简易红烧肉， 每个文档分成很多快。 当我输入红烧肉选择时候，因为知识库有limit，所以只能感知到一些包含红烧肉居多的chunk。 用户问选择哪种红烧肉做法，实际只有4选一，感知不到宏观东西。

**方法一**：
- 增加全文文档向量，与分块 建立父子关系。 检索到chunk后与父亲文档一起带回 构造上下文。
    - 疑问： 这可能会产生很长内容，且父亲内容由于与子chunk重复，context会有一些重复信息，这应该是没有用的。而且这可能回归到刚开始直接不分块。 
- 因为知识库是topk检索，还是会有一部分没有找到。 
    - 疑问： 虽然可能还不够，但这应该是允许的。
    - 通过查询重写可以改善，让查询更加精确,避免短的模糊的
        - 此外，可以产生多个重写查询。对milvus多路进行查询，search就会更加可靠

方法二：
知识图谱。（未实现）

此外，我们引入稀疏向量，允许对菜谱种一些定量信息保持准确 敏感 完整

基于baseline代码修改

## baseline不变代码

In [20]:
import torch
import glob
import os
from dotenv import load_dotenv
import os
from typing import Any
import uuid

cuda_available = torch.cuda.is_available()
print(f"是否支持 CUDA (GPU加速): {cuda_available}")

load_dotenv()
print(os.getcwd())

是否支持 CUDA (GPU加速): False
/home/dong/data-analysis/myrag/cookrag


In [21]:
COLLECTION_NAME = 'cookrag_ct_v1' # 新的知识库

In [23]:
from pymilvus import MilvusClient
client = MilvusClient(uri='milvus_cookrag.db')

In [55]:
def build_context(knowledges: list[dict[str, Any]])->str:
    """ 知识库返回结果转换位上下文 """
    context = ""
    print(f'context get {len(knowledges)} articles')
    for kd in knowledges:
        context+= kd['entity']['content']
        print(f'context- source :{kd['entity']['metadata']['source']}')
    return context

PromptTemplate 产生纯字符串。
ChatPromptTemplate 产生携带角色的消息列表，

In [51]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    api_key= os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
    model='inclusionai/ling-2.6-1t:free',
) 

In [48]:
# llm = ChatOpenAI(
#         api_key=os.getenv('AIHUBMIX_API_KEY_XIAOMI'),
#         base_url="https://aihubmix.com/v1",
#         model='xiaomi-mimo-v2-omni-free'
#     ) 

In [57]:
from langchain_core.prompts import ChatPromptTemplate

def rag(query:str)->str:
    """ 用户接口 """
    template = ChatPromptTemplate(
        messages=[
            ('system', '你是一个专业的烹饪饮食专家, 严格根据上下文信息回答，如果不知道就说不知道'),
            ('human', """
            上下文信息：{context} 
                    
            用户输入:{query}
            """)
                ]
    )
    chain = template | llm
    rewrite_queries = query_rewrite(user_query=query)
    print(f'rewrite queries: {rewrite_queries}')
    knowledges = knowledge_search(queries=rewrite_queries)
    context = build_context(knowledges=knowledges)
    response = chain.invoke({'context': context, 'query':query})
    return response.content

In [27]:
from IPython.display import display, Markdown
def display_md_jupyter(s:str):
    """ jupyter是纯文本输出，使用IPython交互内置渲染md """
    display(Markdown(s))

## 修改

我们在`metadata`字段中，为每个全文文档设置`id`,为每个chunk子文档设置`parent_id`,这两个相等关联

In [28]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader

def load_documents() -> list[Document]:
    """ 将每个md文档加载为对应document结构 ，添加id辅助后面关联"""
    documents = []
    for md_file in glob.glob(os.path.join(os.getcwd(), 'data', 'cooking','*.md')):
        loader = TextLoader(file_path=md_file)
        data = loader.load()
        data = data[0]
        data.metadata['id'] = str(uuid.uuid4())
        documents.append(data)
    return documents

In [29]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

def splitter_documents(documents: list[Document]) -> list[Document]:
    """ 对一些document分块 """
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
    
    text_splitter = RecursiveCharacterTextSplitter(
        separators = ["\n\n", "\n", "。", "，", " ", ""],  # 分隔符优先级
        chunk_size = 200,
        chunk_overlap=10
    )
    all_sections = []
    
    for doc in documents:
        sections  = markdown_splitter.split_text(doc.page_content)
        for section in sections:
            section.metadata['parent_id'] = doc.metadata['id']
        all_sections.extend(sections)

    chunks = text_splitter.split_documents(all_sections)

    # page_content会丢失层级信息，只保留内容. 
    # 这导致一勺盐不知道是红烧肉还是豆腐
    # 需要手动为其注入
    for chunk in chunks:
        prefix = ""
        h1 = chunk.metadata.get('Header 1', '无')
        h2 = chunk.metadata.get('Header 2', '无')
        h3 = chunk.metadata.get('Header 3', '无')
        prefix = f"主题: {h1} > 章节: {h2} > 细节: {h3}\n内容: "
        chunk.page_content = prefix + chunk.page_content 
    return chunks

In [ ]:
from pymilvus import MilvusClient,FieldSchema, CollectionSchema,DataType
def setup_collection():
    if client.has_collection(COLLECTION_NAME):
        client.drop_collection(COLLECTION_NAME) 
    # 定义数据库格式
    fileds = [
        FieldSchema(name='id', dtype=DataType.INT64, is_primary=True,auto_id = True),
        FieldSchema(name='content', dtype=DataType.VARCHAR, max_length=1024),
        FieldSchema(name='metadata', dtype=DataType.JSON),
        FieldSchema(name='sparse_vector', dtype=DataType.SPARSE_FLOAT_VECTOR),
        FieldSchema(name='dense_vector', dtype=DataType.FLOAT_VECTOR, dim=embed_encoder.dim['dense']),
    ]
    schema = CollectionSchema(fields=fileds, description='cookrag')
    client.create_collection(collection_name=COLLECTION_NAME, schema=schema)
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name='dense_vector',
        index_type='IVF_FLAT', # 索引类型, 稠密向量
        metric_type='IP',  # 
    )
    client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name='sparse_vector',
        index_type='SPARSE_INVERTED_INDEX', # 索引类型。稀疏向量
        metric_type='IP',  # 必须显式指定为内积

    )
    client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

In [31]:
from scipy.sparse import coo_array, coo_matrix
def format_sparse_vector_to_dict(coo_arr: coo_array) -> dict:
    """ 
    将 coo_array 数组 转换为 Milvus 稀疏向量字段要求的字典格式: 
    {index: value}
    """
    # 1D coo_array 的坐标在 coords[0] 中
    indices = coo_arr.coords[0] 
    return dict(zip(indices, coo_arr.data))


In [32]:
def insert_collection(documents:list[Document])->None:
    """ 把docuemnts 放到向量数据库 """
    data_to_insert = []

    for doc in documents:
        embeddings = embed_encoder([doc.page_content])
        data_to_insert.append({
            'metadata': doc.metadata,
            'dense_vector': embeddings['dense'][0],
            'sparse_vector': format_sparse_vector_to_dict(embeddings['sparse'][0]),
            'content': doc.page_content
        })
    client.insert(
        collection_name=COLLECTION_NAME,
        data = data_to_insert
    )

In [33]:
def build_collection():
    """ 构建向量知识库 """
    setup_collection()
    print('collection 结构已准备好')
    documents = load_documents()
    splitted_documents = splitter_documents(documents)
    insert_collection(documents)
    print('插入全文文档向量 ok')
    insert_collection(splitted_documents)
    print('插入文本块ok')

In [34]:
# build_collection() 

In [39]:
from pymilvus import RRFRanker,AnnSearchRequest
def knowledge_search(queries: list[str]) -> list[dict[str, Any]]:
    """ 混合检索 稀疏向量和稠密向量, 多路搜索 """
    hybrid_results = []# {'id_value': r}
    for query in queries:
        query_embeddings = embed_encoder([query])
        dense_vec, sparse_vec = query_embeddings['dense'][0], format_sparse_vector_to_dict(query_embeddings['sparse'][0])
        # RRF
        rerank = RRFRanker(k=60) # 会融合两个结果，得到, k=60表示平滑参数（就是两个结果平衡参数）
        dense_req = AnnSearchRequest(
            [dense_vec],
            anns_field='dense_vector',
            limit=30,
            param={"metric_type": "IP"}
        )
        sparse_req = AnnSearchRequest(
            [sparse_vec],
            anns_field='sparse_vector',
            limit=30,
            param={"metric_type": "IP"}
        )
        results = client.hybrid_search(
            collection_name=COLLECTION_NAME,
            reqs=[dense_req, sparse_req],
            ranker=rerank,
            limit=30,
            output_fields=['content','metadata']
        )[0]
        for r in results:
            hybrid_results.append(r)

    # 只返回父亲主页文档，否则导致知识重复，浪费上下文。
    need_ids = set()
    for r in hybrid_results:
        parent_id = r['metadata'].get('parent_id', '')
        # 如果由parentid 那就是子chunk
        if parent_id:
            need_ids.add(parent_id)
        # 如果有id 那就是父亲
        id = r['metadata'].get('id', '')
        if id:
            need_ids.add(id)
    final_results = []
    if need_ids:
        pid_list = list(need_ids)
        # 准确查询
        parent_results = client.query(
            collection_name=COLLECTION_NAME,
            filter=f"metadata['id'] in {pid_list}",
            output_fields=['content','metadata']
        )
        # 可以考虑不要格式化，因为现在都是返回父chunK,
        formatted_parents = []
        for p in parent_results:
            formatted_parents.append({
                'id': p['id'],
                'distance': 0.0, # 补齐字段，防止后续代码报错
                'entity': {
                    'content': p['content'],
                    'metadata': p['metadata']
                }
            })
        final_results.extend(formatted_parents)
    return final_results

增加查询重写，针对红烧肉这种短查询。

需要调用llm

In [61]:
from langchain_core.prompts import ChatPromptTemplate

def query_rewrite(user_query: str):
    """
    很重要！
    """
    chat_template = ChatPromptTemplate.from_messages([
        ("system", (
            """
            你是一个烹饪百科专家。请对用户的查询进行重写。

            原则:
            1. 不要对用户查询进行回答！！你的任务是帮助RAG系统优化查询，而不是回答用户问题。
            2. 多维发散：如果查询含糊，请从“做法、搭配、场景、专业术语”等角度把关键词拓宽。
            3. 语义逻辑关联：如番茄与西红柿，洋芋和土豆，

            示例：
            输入：番茄可以做什么
            输出：
            番茄家常菜做法大全
            西红柿配肉类/海鲜的烹饪技巧
            番茄口味的汤羹与调味汁配方
            西红柿作为主料的创意料理
            番茄营养搭配与烹饪禁忌

            请重写用户输入：{{User_Query}}
            输出的重写查询按照换行分割
            """
        )),
        ("human", "原始问题：{user_query}")
    ])
    
    # 链式调用
    chain = chat_template | llm
    
    response = chain.invoke({"user_query": user_query})
    # 处理结果
    rewritten_queries = [q.strip() for q in response.content.split('\n') if q.strip()]
    return list(set(rewritten_queries))


In [63]:
rewrite_queries = query_rewrite(user_query='有没有那种适合搭配这种红色酸味蔬菜的白肉类食材？')
rewrite_queries

['西红柿与白肉在烹饪中的火候与调味逻辑',
 '番茄类食材搭配白肉的场景（宴客、减脂、快手菜）',
 '红色酸味蔬菜与白肉类组合的家常做法',
 '番茄白肉食材搭配菜谱',
 '西红柿配鸡/鸭/猪肉的去腥提鲜技巧']

In [64]:
rewrite_queries = query_rewrite(user_query='跟番茄炒蛋用的同一种增味调料的川菜有哪些？')
rewrite_queries

['川菜里用白糖+酱油/醋提味的热菜与家常小炒',
 '番茄炒蛋增味调料在川式凉菜、炒菜、烧菜中的迁移应用',
 '用与番茄炒蛋相同增味调料复刻川式风味：调料组合与菜例',
 '番茄炒蛋调味逻辑（酸甜提鲜）在川菜里的对应做法（如鱼香、荔枝味型）',
 '番茄炒蛋常用增味调料（如白糖、酱油/生抽、料酒、香葱等）的川菜应用',
 '川菜中与番茄炒蛋同类型增味调料（糖、酱油、醋、葱姜蒜）的经典菜品']

🙌🙌可以看到，对于这种反向、多级的，如果是内部知识库，是很难重写的。这就要知识图谱！

In [42]:
results = knowledge_search(rewrite_queries)

In [43]:
results[0]

{'id': 466035963454291994,
 'distance': 0.0,
 'entity': {'content': '# 红烧鱼头的做法\n\n- **WARNING** 如果没有使用过菜刀剁过肉类食物，那么并不推荐使用该菜单！！！\n- 在操作中，锋利的菜刀可能会划伤手指，请一定要小心。\n\n预估烹饪难度：★★★★\n\n## 必备原料和工具\n\n- 注：如果有可能，尽量另准备一把菜刀，超市或市场上均有廉价且刀片更厚的菜刀，刀片厚度在 5-7mm 为最佳。\n- 大葱、姜、大蒜、香菜、美人椒\n- 油、盐、鸡精、生抽、老抽、陈醋、黑胡椒粉、料酒\n- 八角、干辣椒\n- 鱼头一个\n- 注：市场直接贩卖的鱼头一般分为两种：白鲢、花鲢。前者价格便宜，后者价格略贵，但口感也更佳！\n\n## 计算\n\n注意，这道菜仅有足够 2 人食用的版本。\n\n* 鱼头一个\n* 大葱 200g\n* 姜 80g\n* 蒜瓣 3-4 个\n* 美人椒 1/4 个\n* 香菜 4 棵\n* 八角两个，干辣椒五个\n\n## 操作\n\n### 原材料准备\n\n* 葱、姜、蒜、香菜、美人椒分别清洗干净。\n* 干辣椒与八角稍微冲洗即可。\n* 大葱切两半。后半段大葱（葱白处）切段，每段长度约 4cm。前半段（葱叶处）先切段，再将每段劈为四瓣。\n* 姜切片，每片厚度约 3mm。\n* 大蒜拍碎。\n* 拿出两棵香菜去根，切为 1.5cm 香菜碎。\n* 将美人椒切为厚度为 3mm 的辣椒圈。\n* 干辣椒切四段。\n\n### 腌制鱼头\n\n* 注：下文所述的鱼身是购买鱼头时所附带的鱼肉。\n* 将鱼头去鳞，清洗鱼头处未被清理干净的内脏。\n* 剁去鱼鳍、清理鱼鳃。\n* 将鱼头下巴与鱼身连接的地方剁开，鱼身剁块，鱼头剁成四/六瓣。\n  * 注：鱼的处理很难用文字完全表述，可以搜索鱼头处理相关视频。\n* 将剁好的鱼头进行清洗，最好洗掉鱼块上滞留的血水。\n* 将清洗好的鱼块放入盆中，加入 5g 盐、10g 生抽、10g 料酒。放入葱（前半段切碎的那个）、1/3 姜片。将其拌匀，静置 1-2 小时。\n\n### 最终步骤\n\n* 加入 30ml 油，等待锅热...\n* 油热，将锅关至小火\n  * 如果不明白为何要这样做，请查看[学习炒与煎](.

In [ ]:
def sparse_vec_info(s:str):
    """ from ai. 描述稀疏向量激活了多少关键词 """
    embeddings = embed_encoder([s])
    sparse_obj = embeddings['sparse'][0]
    print(f"类型: {type(sparse_obj)}")
    print(f"非零元素个数: {sparse_obj.nnz}") # 查看有多少个关键词被激活
    tokenizer = embed_encoder.model.tokenizer
    coo = sparse_obj.tocoo()
    token_weights = sorted(zip(coo.col, coo.data), key=lambda x: x[1], reverse=True)
    
    print(f"{'关键词':<10} | {'权重':<10}")
    print("-" * 25)
    for token_id, weight in token_weights:
        token_text = tokenizer.decode([token_id]) # 将 ID 转回文字
        print(f"{token_text:<10} | {weight:.4f}")

可以看到，对于短查询，激活的关键词很少很少。大部分只是把每个字拆开。  这完全不包含信息，是完全的噪声

- 我们发现，对于红烧肉稀疏向量，只有3个非0。 对于一大段话，大部分都是单字，且产生很多非0.
- 做度量时候，内积，就是在这 红 烧 肉 三个字进行。 
- 所以文字越多的文章，越可能相近的。

所以我们会设置更多limit，让RRF过滤。 
RRF是统计模型，所以不能无限制增大limit,会增大噪声。

西红柿对于 红烧肉 的稀疏得分很高， 这是可以理解的。是允许的。

如果需要更改双方得分比重：rerank = WeightedRanker(0.8, 0.2) 

## 测试

In [ ]:
display_md_jupyter(rag('对比一下南派和徽派红烧肉在甜度控制上的区别。'))

😊😊😊
- 可以看到，相比于之前，有了丰富的上下文。因此可以回答
- 用量明确，稀疏向量也起了作用

In [ ]:
display_md_jupyter(rag('我想做红烧肉，但我家只有白砂糖没有冰糖，哪种做法最适合我'))

😊😊😊
回答是合理的

In [ ]:
display_md_jupyter(rag('红烧肉怎么上色'))

😊😊😊
回答是合理的。 LLM还矫正了料酒不上色，并没有幻觉

In [ ]:
display_md_jupyter(rag('如何制作无骨鸡爪'))

😊😊😊对于需要详细制作的。LLM只是让语句变流畅了，没有改变

In [ ]:
display_md_jupyter(rag('给我推荐最高难度川菜'))

😶😊是合理的，因为本身没有答案，这个只要菜在知识库。由LLM选择

In [ ]:
display_md_jupyter(rag('番茄有什么营养价值'))

😊😶😊可以看到，LLM理解了上下文，出于prompt，他有着明显的限制

In [ ]:
display_md_jupyter(rag('鸡爪'))

😊

In [58]:
display_md_jupyter(rag('番茄可以做什么菜'))

rewrite queries: ['番茄炒虾仁', '番茄鸡蛋汤', '番茄炒蛋', '番茄凉拌黄瓜', '番茄炖牛腩', '番茄鱼片汤', '番茄意面', '番茄蒸蛋']


I0000 00:00:1777799022.997585    2332 chttp2_transport.cc:1182] unix:/tmp/tmpyhkknj8r_milvus_cookrag.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {created_time:"2026-05-03T17:03:42.99754198+08:00", http2_error:11, grpc_status:14}
E0000 00:00:1777799022.997778    2332 chttp2_transport.cc:1210] unix:/tmp/tmpyhkknj8r_milvus_cookrag.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


context get 48 articles
context- source :/home/dong/data-analysis/myrag/cookrag/data/1Star.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/芥末黄油罗氏虾.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/罗宋汤.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/红芸豆拌饭.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/芥末罗氏虾.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/干煎阿根廷红虾.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/西红柿鸡蛋挂面.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/炒茄子.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/西红柿土豆炖牛肉.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/黄油鸡.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/印度奶茶.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/印度葫芦丸子.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/糖拌西红柿.md
context- source :/home/dong/data-analysis/myrag/cookrag/data/西红

根据上下文信息，番茄可以做以下菜品（按类别整理）：

## 素菜类
- **西红柿炒鸡蛋**（经典家常菜，番茄+鸡蛋）
- **西红柿豆腐汤羹**（番茄+豆腐+鸡蛋，清淡汤羹）
- **番茄红酱**（可作为意面、薄饼等主食的酱料）
- **糖拌西红柿**（凉拌菜，番茄+白糖）
- **西红柿鸡蛋汤**（简单汤品）
- **地三鲜**（番茄不是主料但可作为配料之一）
- **茄子炖土豆**（同上，可作为配料）

## 荤菜类
- **西红柿牛腩**（番茄+牛腩，汤汁浓厚）
- **西红柿土豆炖牛肉**（番茄+牛肉+土豆）
- **番茄牛肉蛋花汤**（番茄+牛肉+鸡蛋）

## 主食类
- **西红柿鸡蛋挂面**（番茄+鸡蛋+挂面）
- **意式肉酱面**（番茄酱是意面酱的重要成分）
- **意大利面**（可搭配番茄红酱）

## 汤类
- **罗宋汤**（番茄是重要配料，搭配牛肉、蔬菜）
- **奶油蘑菇汤**（部分版本会加番茄）
- **各种蔬菜汤**（番茄常作为提味配料）

## 其他类别
- **多种咖喱**（如印度黄油鸡、牛肉咖喱等，番茄常作为咖喱酱的基底）
- **焖饭**（如印度焖饭，番茄可增加风味）
- **沙拉**（番茄是常见沙拉食材）
- **饮品**（如番茄汁、番茄特调等）

---

**重点推荐**（按制作简便程度排序）：
1. **西红柿炒鸡蛋**（最简单，5分钟完成）
2. **西红柿鸡蛋挂面**（15分钟，主食）
3. **西红柿豆腐汤羹**（10分钟，汤品）
4. **番茄红酱**（可一次多做冷冻保存）
5. **西红柿牛腩**（稍复杂但味道好）

如果需要具体某道菜的做法，可以告诉我菜名，我会提供详细步骤！

因为增加了查询重写。